# Marvel Comics Network Analysis

Analyzing the Wikipedia Marvel Comics superhero network (week 1 dataset).
- 303 characters
- 1,784 directed edges
- Data source: Wikipedia Category:Marvel Comics superheroes

In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load Marvel edges data
df_edges = pd.read_csv('../data/week1_edges.tsv', sep='\t', skiprows=5, names=['source', 'target'])
print(f"Loaded {len(df_edges)} edges")
print(f"\nFirst 10 edges:")
print(df_edges.head(10))

# Also load nodes data for enrichment
df_nodes = pd.read_csv('../data/week1_nodes.tsv', sep='\t', skiprows=4)
print(f"\n\nLoaded {len(df_nodes)} nodes")
print(f"Node columns: {list(df_nodes.columns)}")

In [ ]:
# Create directed graph
G = nx.DiGraph()
for _, row in df_edges.iterrows():
    G.add_edge(row['source'], row['target'])

# Also add isolated nodes from nodes file
for node_id in df_nodes['node_id']:
    if node_id not in G:
        G.add_node(node_id)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

In [ ]:
# Network statistics
print("Network Statistics:")
print(f"Density: {nx.density(G):.3f}")
print(f"Strongly connected: {nx.is_strongly_connected(G)}")
print(f"Strongly connected components: {nx.number_strongly_connected_components(G)}")

In [ ]:
# Degree analysis with character names
degrees = [(n, G.in_degree(n), G.out_degree(n)) for n in G.nodes()]
degrees_df = pd.DataFrame(degrees, columns=['node_id', 'in_degree', 'out_degree'])
degrees_df['total_degree'] = degrees_df['in_degree'] + degrees_df['out_degree']

# Merge with node names
degrees_df = degrees_df.merge(df_nodes[['node_id', 'name']], on='node_id', how='left')

print("Top 10 Marvel characters by total degree:")
top_10 = degrees_df.nlargest(10, 'total_degree')[['name', 'in_degree', 'out_degree', 'total_degree']]
print(top_10.to_string(index=False))

In [ ]:
# Export for visualization
# Convert graph to JSON format for web visualization
import json

nodes = [{'id': str(node)} for node in G.nodes()]
edges = [{'source': str(u), 'target': str(v)} for u, v in G.edges()]

graph_json = {'nodes': nodes, 'edges': edges}

with open('../docs/data/network.json', 'w') as f:
    json.dump(graph_json, f)

print(f"Exported graph to network.json")